In [ ]:
import dlib
print("✅ dlib version :", dlib.__version__)


✅ dlib version : 19.22.99


In [ ]:
import face_recognition
print("✅ face_recognition version :", face_recognition.__version__)


✅ face_recognition version : 1.2.3


In [4]:
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
cfg.MODEL.DEVICE = "cpu" # This guarantees that even if your code later tries .to("cuda"), it will stay on CPU and won’t crash.

predictor = DefaultPredictor(cfg)  # By default, will use CPU if CUDA unavailable


In [10]:
#verifier l'installation de toutes les dépendances
import cv2
print("OpenCV version :", cv2.__version__)
img = cv2.imread("./image.jpg")
print("Lecture image :", "OK" if img is not None else "Échec (image non trouvée)")
#####
import torch
print("PyTorch version :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))
#####

import dlib, face_recognition
print("dlib version :", dlib.__version__)
print("face_recognition version :", face_recognition.__version__)

# Test simple de reconnaissance (sans image)
print("Nombre de visages détectés sur une image factice :", len(face_recognition.face_encodings(face_recognition.load_image_file("./Jim_Carry_RGB.webp"))) if "./Jim_Carry_RGB.webp" else "Test image non trouvée")


OpenCV version : 4.12.0
Lecture image : OK
PyTorch version : 2.9.0+cpu
CUDA disponible : False
dlib version : 19.22.99
face_recognition version : 1.2.3


RuntimeError: Unsupported image type, must be 8bit gray or RGB image.

In [ ]:
# ========== EXERCICE 1 : Analyse d'une image ==========

# Lecture et affichage d'une image
import cv2
img = cv2.imread("./image.jpg")
cv2.imshow("Image", img)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [ ]:
# Conversion en niveaux de gris
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
cv2.imshow("Niveaux de gris", gray)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Détection des contours
edges = cv2.Canny(gray, 100, 200)
cv2.imshow("Contours Canny", edges)
cv2.waitKey(0)
cv2.destroyAllWindows()



In [ ]:

# ========== EXERCICE 2 : Lecture vidéo / webcam ==========

import cv2

cap = cv2.VideoCapture(0)  # 0 pour webcam, ou nom du fichier vidéo

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Convertir en niveaux de gris
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Afficher la vidéo
    cv2.imshow("Webcam - Niveaux de gris", gray)
    
    # Appuyer sur 'q' pour quitter
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:


# ========== EXERCICE 3 : Détection d'objets avec Detectron2 ==========

# Installation Detectron2 (Google Colab)
# !pip install -U 'git+https://github.com/facebookresearch/detectron2.git'

# Uploader une image
# from google.colab import files
# uploaded = files.upload()

import cv2
import torch
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog
import matplotlib.pyplot as plt


# Configuration du modèle pré-entraîné
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
)
predictor = DefaultPredictor(cfg)

# Charger une image
img = cv2.imread("image1.jpg")

# Prédiction
outputs = predictor(img)

# Visualisation
v = Visualizer(img[:, :, ::-1],
               MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))

# Affichage
plt.figure(figsize=(12, 8))
plt.imshow(out.get_image()[:, :, ::-1])
plt.axis('off')
plt.show()


In [ ]:

# ========== EXERCICE 4 : Détection faciale avec Dlib ==========

# Installation
# pip install dlib face_recognition

import dlib
import cv2

detector = dlib.get_frontal_face_detector()
img = cv2.imread("people.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
faces = detector(gray)

for face in faces:
    x, y, w, h = face.left(), face.top(), face.width(), face.height()
    cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)

from google.colab.patches import cv2_imshow
cv2_imshow(img)


In [ ]:


# ========== EXERCICE 5 : Reconnaissance faciale ==========

# Installation des dépendances (Google Colab)
# !apt-get install -y cmake libopenblas-dev liblapack-dev
# !pip install dlib face_recognition

import face_recognition
import os

# Désactiver CUDA pour dlib
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# Charger les images
image_ref = face_recognition.load_image_file("Jim_Carrey_2008.jpg")
encodage_ref = face_recognition.face_encodings(image_ref)[0]

image_test = face_recognition.load_image_file("Jim_Carrey_2010.jpg")
encodage_test = face_recognition.face_encodings(image_test)[0]

# Comparer
result = face_recognition.compare_faces([encodage_ref], encodage_test)
print("Même personne ?", result[0])